# Statistical Analysis: Census Income

**Author:** Belal Nwiran

**Dataset:** [Census Income](https://archive.ics.uci.edu/dataset/20/census+income)

## Purpose
This notebook analyzes a census dataset of 32,561 adults and 15 columns, including age, education, occupation, hours worked per week, sex, and income level (`<=50K` or `>50K`).

## Analytical Question
Is income level associated with sex in this dataset?

- **H0:** Income level and sex are independent.
- **H1:** Income level and sex are not independent.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset Loading

In [2]:
cols = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week",
        "native_country", "income"]

df = pd.read_csv("adult.data", header=None, names=cols,
                 skipinitialspace=True, na_values="?")
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
df.shape

(32561, 15)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [5]:
df.duplicated().sum()

np.int64(24)

In [6]:
def drop_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drop exact duplicate rows (all columns) and report how many were removed."""
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates(keep="first")
    df = df.reset_index(drop=True)
    print(f"Dropped {before - len(df)} duplicate rows ({before} -> {len(df)})")
    return df

In [7]:
df = drop_duplicate_rows(df)

Dropped 24 duplicate rows (32561 -> 32537)


In [8]:
df.isna().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     582
income               0
dtype: int64

## Dataset Loading Conclusion

* **Size:** There are 32561 rows and 15 columns.
* **Data types:** All columns have the correct data types.
* **Duplicated rows:** 24 rows.
* **Missing values:** There are missing values in:
  1. `workclass`: 1836 (5.64%)
  2. `occupation`: 1843 (5.66%)
  3. `native_country`: 582 (1.79%)

# Descriptive Statistics

In [9]:
df.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32537.000000,3.253700e+04,32537.000000,32537.000000,32537.000000,32537.000000
mean,38.585549,1.897808e+05,10.081815,1078.443741,87.368227,40.440329
std,13.637984,1.055565e+05,2.571633,7387.957424,403.101833,12.346889
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.369930e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [10]:
df.describe(include="str")

,workclass,education,marital_status,occupation,relationship,race,sex,native_country,income
count,30701,32537,32537,30694,32537,32537,32537,31955,32537
unique,8,16,7,14,6,5,2,41,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,22673,10494,14970,4136,13187,27795,21775,29153,24698


In [11]:
for col in ["sex", "income", "race", "workclass"]:
    print(df[col].value_counts(dropna=False))
    print()

sex
Male      21775
Female    10762
Name: count, dtype: int64

income
<=50K    24698
>50K      7839
Name: count, dtype: int64

race
White                 27795
Black                  3122
Asian-Pac-Islander     1038
Amer-Indian-Eskimo      311
Other                   271
Name: count, dtype: int64

workclass
Private             22673
Self-emp-not-inc     2540
Local-gov            2093
NaN                  1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64



In [12]:
df["income"].value_counts(normalize=True)

income
<=50K    0.759074
>50K     0.240926
Name: proportion, dtype: float64

In [13]:
pd.crosstab(df["sex"], df["income"], normalize="index")

income,<=50K,>50K
sex,,
Female,0.890448,0.109552
Male,0.694145,0.305855


In [14]:
(df["capital_gain"] == 0).mean(), (df["capital_loss"] == 0).mean()

(np.float64(0.9166487383594062), np.float64(0.9533146878937825))

In [15]:
(df["capital_gain"] == 99999).sum()

np.int64(159)

In [16]:
big = df[df["capital_gain"] == 99999]
big["income"].value_counts(normalize=True)

income
>50K    1.0
Name: proportion, dtype: float64

In [17]:
big["sex"].value_counts(normalize=True)

sex
Male      0.861635
Female    0.138365
Name: proportion, dtype: float64

## Descriptive Statistics

### Numeric variables
The dataset, after removing duplicates, has 32,537 rows. The numeric columns have no missing values.

- **Age:** The average age is 38.6 and the median is 37. Ages go from 17 to 90, and the middle half of people are between 28 and 48.
- **Hours per week:** The median is 40 and the middle half is between 40 and 45. Most people work about 40 hours. The values go from 1 to 99.
- **Education years:** The median is 10 and the middle half is between 9 and 12. This means most people have about a high-school level of education.
- **Capital gain and capital loss:** The median and the 75th percentile are both 0, so most people have none. Zero values are 91.67% of capital gain and 95.33% of capital loss. The standard deviation of capital gain (about 7,385) is much larger than its mean (about 1,078). A small number of large values pull the mean up, so the mean is not a good summary for this column.
- **`fnlwgt`:** This is a census sampling weight, not a measure of a person. I do not use it in the analysis.

### Categorical variables
- **Sex:** 21,775 people are male (about 66.9%) and the rest are female (about 33.1%). The groups are not balanced.
- **Income:** 24,698 people (about 75.9%) earn `<=50K` and about 24.1% earn `>50K`.
- **Race:** White is the largest group with 27,795 people (about 85.4%), so one group dominates the sample.
- **Other columns:** The most common values are `HS-grad` for education (about 32.3%), `Married-civ-spouse` for marital status (about 46.0%), `Private` for workclass (about 73.9% of non-missing values), and `United-States` for native country (about 91.2% of non-missing values).

### Missing values
Missing values appear only in three categorical columns: `workclass` (1,836), `occupation` (1,843) and `native_country` (583). The columns I test, `sex` and `income`, have no missing values, so I do not remove or fill any rows.

### Income by sex (preview of the test)
In this sample, 10.9% of women and 30.6% of men earn `>50K`. This is a large difference. It only describes the data. It does not show that sex causes the difference, because other factors such as age, hours worked or occupation may also play a role. The formal test comes later.

### Extreme capital gains
159 rows have a capital gain of 99999. All of them are in the `>50K` group, and about 86% are male, compared with about 67% male in the full data. The dataset documentation does not explain these extreme values. Removing these rows would delete only high-income people and change the counts I am testing, so I keep them.